In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold, train_test_split
from sklearn.metrics import classification_report, f1_score

# Configuración de rutas
INPUT_PATH = '../../data/processed/dataset_aireado_processed.csv'
MODEL_PATH = '../../models/artifacts/'

if not os.path.exists(MODEL_PATH):
    os.makedirs(MODEL_PATH)

In [2]:
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Dataset cargado: {df.shape}")
else:
    print(f"Error: No se encuentra el archivo en {INPUT_PATH}")

# Definir columnas de control y target
target = 'fault_id'
ignore_cols = ['run_id', 'time_min', 'fault_id', 'fault']
features = [col for col in df.columns if col not in ignore_cols]

print(f"Features a utilizar ({len(features)}): {features[:5]}... entre otras.")

Dataset cargado: (12000, 41)
Features a utilizar (38): ['T_amb', 'T_set', 'Kg_embutido', 'N_fan_Hz', 'T_cab']... entre otras.


In [3]:
# Separamos los Run IDs para que el test sea totalmente independiente
unique_runs = df['run_id'].unique()
train_runs, test_runs = train_test_split(unique_runs, test_size=0.2, random_state=42)

df_train = df[df['run_id'].isin(train_runs)].copy()
df_test = df[df['run_id'].isin(test_runs)].copy()

X_train = df_train[features]
y_train = df_train[target]
groups_train = df_train['run_id']

X_test = df_test[features]
y_test = df_test[target]

print(f"Entrenamiento: {len(df_train)} muestras ({len(train_runs)} ciclos)")
print(f"Test: {len(df_test)} muestras ({len(test_runs)} ciclos)")

Entrenamiento: 9600 muestras (240 ciclos)
Test: 2400 muestras (60 ciclos)


In [4]:
# Definición del espacio de búsqueda de hiperparámetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False],
    'criterion': ['gini', 'entropy']
}

# Usamos GroupKFold para que un "run_id" nunca esté en train y val a la vez
gkf = GroupKFold(n_splits=5)

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

# Configuramos la búsqueda
rf_random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=10, # Ajustar según capacidad de cómputo
    cv=gkf,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    scoring='f1_weighted'
)

In [5]:
print("--- BÚSQUEDA DE HIPERPARÁMETROS ---")
# IMPORTANTE: Pasamos el argumento 'groups' para el GroupKFold
rf_random_search.fit(X_train, y_train, groups=groups_train)

print("\nProceso completado.")

--- BÚSQUEDA DE HIPERPARÁMETROS ---
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Proceso completado.


In [6]:
print("==============================================")
print("  RESULTADOS DEL TUNING (BEST PARAMETERS)     ")
print("==============================================")
print(f"Mejor F1-Score (CV): {rf_random_search.best_score_:.4f}")
print("\nConfiguración óptima:")
for param, value in rf_random_search.best_params_.items():
    print(f" - {param}: {value}")

# Guardar el mejor estimador
best_rf = rf_random_search.best_estimator_
joblib.dump(best_rf, os.path.join(MODEL_PATH, 'rf_aireado_tuned.pkl'))
print(f"\n Modelo guardado en: {MODEL_PATH}rf_aireado_tuned.pkl")

  RESULTADOS DEL TUNING (BEST PARAMETERS)     
Mejor F1-Score (CV): 1.0000

Configuración óptima:
 - n_estimators: 200
 - min_samples_split: 2
 - min_samples_leaf: 4
 - max_depth: None
 - criterion: entropy
 - bootstrap: False

 Modelo guardado en: ../../models/artifacts/rf_aireado_tuned.pkl


In [7]:
# Evaluamos sobre el 20% que nunca vio el tuning (Hold-out test)
y_pred = best_rf.predict(X_test)

print("--- REPORTE DE RENDIMIENTO EN TEST (SIN POST-PROCESO) ---")
print(classification_report(y_test, y_pred))

--- REPORTE DE RENDIMIENTO EN TEST (SIN POST-PROCESO) ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1720
           1       1.00      1.00      1.00       200
           2       1.00      1.00      1.00       200
           3       1.00      1.00      1.00       280

    accuracy                           1.00      2400
   macro avg       1.00      1.00      1.00      2400
weighted avg       1.00      1.00      1.00      2400

